In [1]:
import pandas as pd
import re
from dsp_nesta_brain import PROJECT_DIR

from bs4 import BeautifulSoup
import json

SITEMAP_PATH = PROJECT_DIR / 'data/Nesta_sitemap_2024-10-29.csv'

BASE_URL = "https://nesta.org.uk"

In [2]:
import requests
from scraping import scrape

In [3]:
def split_url(url: str) -> list:
    """Split a URL into its components"""
    try:
        return [u for u in url.split('/') if len(u) > 0]
    except:
        return None

def url_to_uid(url: str) -> str:
    """Convert a URL to a UID by removing the base URL and replacing slashes with dashes"""
    uid = url.replace(BASE_URL, "").replace("/", "-")
    uid = re.sub("-$", "", uid)
    return uid

def clean_uid(uid: str) -> str:
    """Remove dash in the beginning of the UID"""
    return re.sub("^-", "", uid)

In [4]:
sitemap_df = (
    pd.read_csv(SITEMAP_PATH)
    # Get the url components
    .assign(site_group = lambda df: df.website.apply(split_url))
    .dropna(subset=['site_group'])
    # Remove home page from the list
    .assign(site_group_len = lambda df: df.site_group.apply(len))
    .query('site_group_len > 0')
    .drop(columns=['site_group_len'])
    # Get the site group
    .assign(site_group = lambda df: df.site_group.apply(lambda x: x[0]))
    .drop_duplicates(subset=['website'])
    # turn views into integers
    .fillna({'views': '0', 'active_users': '0'})
    .assign(views = lambda df: df.views.apply(lambda x: x.replace(',', '')).astype(int))
    .assign(uid = lambda df: df.website.apply(url_to_uid))
    .assign(uid = lambda df: df.uid.apply(clean_uid))
    # remove duplicated uids
    .sort_values('rank', ascending=True)
    .drop_duplicates(subset=['uid'], keep='first')
)

In [5]:
site_groups_df = (
    sitemap_df
    .groupby('site_group')
    .agg(
        n_urls = ('website', 'count'),
        avg_views = ('views', 'mean'),
        total_views = ('views', 'sum'),
    )
    .sort_values('n_urls', ascending=False)
    .reset_index()
)

In [6]:
SITE_GROUPS = [
    'blog',
    'report',
    'feature',
    'team',
    'project',
    'event',
    'press-release',
    'project_updates',
    'toolkit',
    'data-visualisation-and-interactive',
    'areas_of_work',
    'case-study',
    'about-us',
    'introduction-our-strategy',
    'about',
    'fairer-start',
    'healthy-life',
    'sustainable-future',
]

In [7]:
sites_to_scrape_df = sitemap_df.query("site_group in @SITE_GROUPS")
len(sites_to_scrape_df)

9053

In [8]:
sites_to_scrape_df.query("site_group == 'report'").head(5)

,rank,website,views,active_users,views_per_active_user,avg_engagement_time,site_group,uid
5,6,/report/,27773,"14,442",1.92,47s,report,report
31,32,/report/modelling-ways-to-improve-our-health/,7247,"5,198",1.39,49s,report,report-modelling-ways-to-improve-our-health
35,36,/report/nesta-strategy-2030/,6381,"5,193",1.23,23s,report,report-nesta-strategy-2030
40,41,/report/state-of-the-art-analysing-where-art-m...,5359,"4,318",1.24,23s,report,report-state-of-the-art-analysing-where-art-me...
41,42,/report/heat-pumps-a-user-survey/,4959,"3,261",1.52,33s,report,report-heat-pumps-a-user-survey


- Folder where to store the websites htmls (using uids)
- Folder for PDFs and naming convention for the pdfs
- jsonl file for saving metadata (from the website and pdfs names)
- Scraping script that checks the jsonl and scrapes only missing files (using requests); use asyncio if useful
- Processing script that goes through the files and applies the beautifulsoup script, and saves processed text in... a new jsonl?
- Upload the documents on S3, and write getters for getting them

In [9]:
web = sites_to_scrape_df.iloc[44].website
url = f"{BASE_URL}{web}"
url

'https://nesta.org.uk/report/nesta-standards-of-evidence/'

In [20]:
sites_to_scrape_df.head()

,rank,website,views,active_users,views_per_active_user,avg_engagement_time,site_group,uid
2,3,/team/,106714,"29,243",3.65,1m 03s,team,team
3,4,/about-us/,45303,"35,784",1.27,31s,about-us,about-us
5,6,/report/,27773,"14,442",1.92,47s,report,report
7,8,/introduction-our-strategy/,23743,"17,576",1.35,27s,introduction-our-strategy,introduction-our-strategy
8,9,/fairer-start/,23400,"14,832",1.58,54s,fairer-start,fairer-start


In [18]:
# add my email to the headers
result = requests.get(url, timeout=10, headers={'User-Agent': 'karlis.kanders@nesta.org.uk'})

In [19]:
result.status_code

200

In [16]:
import importlib
importlib.reload(scrape)

<module 'scraping.scrape' from '/Users/karlis.kanders/Code/dsp_nesta_brain/scraping/scrape.py'>

In [17]:
soup = BeautifulSoup(result.text, 'html.parser')  
scrape.extract_data_layer(soup)

[{'areasOfWork': '',
  'missions': '',
  'projects': '',
  'units': 'Alliance for Useful Evidence',
  'authors': 'Joe Ludlow,Ruth Puttick',
  'contentType': 'report page',
  'title': 'Nesta Standards of Evidence',
  'publishDate': '2013-10-29'}]

In [13]:
pdf_links = scrape.extract_pdf_links(soup, BASE_URL)

In [15]:
scrape.download_pdfs(pdf_links)

2024-10-29 15:41:45,302 - dsp_nesta_brain - INFO - Downloaded: standards_of_evidence.pdf


In [150]:
scrape.scrape(url)

{'text': 'This is a toolkit on how to invent, adopt or adapt ideas that can deliver better results. It features 30 practical social innovation tools that are quick to use and simple to apply.\n\nThe toolkit was created by Nesta in partnership with STBY and Quicksand, and made possible by the Rockefeller Foundation.\n\n\n                    The DIY Toolkit\n                \n\nThe free toolkit includes 30 tried and tested social innovation tools, all grounded in existing theories and practices of innovation, design, and business development.\nAs well as the downloadable full toolkit, each of the individual tools is available in a range of sizes as a pdf template for use in the field, so practitioners can dive straight into action.\nThe DIY toolkit website also features a range of case studies and expert blog posts to help people put the tools into practice.\nThe website is also available in Spanish, Arabic, Mandarin, French and Russian.\nThe original toolkit was designed for people work

# Processing data

In [58]:
# Load jsonl file as pandas dataframe
datapath = PROJECT_DIR / 'data/outputs_2024-10-29/metadata.jsonl'
df = (
    pd.read_json(datapath, lines=True)
    .assign(has_pdf = lambda df: df.pdf_files.apply(len) > 0)
)

In [59]:
df

,_status_code,url,website,uid,site_group,rank,views,status_code,pdf_files,web_metadata,pdf_links,has_pdf
0,200,https://nesta.org.uk/event/,/event/,event,event,11,19203,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
1,200,https://nesta.org.uk/team/,/team/,team,team,3,106714,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
2,200,https://nesta.org.uk/sustainable-future/,/sustainable-future/,sustainable-future,sustainable-future,13,16780,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
3,200,https://nesta.org.uk/healthy-life/,/healthy-life/,healthy-life,healthy-life,12,18488,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
4,200,https://nesta.org.uk/report/,/report/,report,report,6,27773,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
5,200,https://nesta.org.uk/about-us/,/about-us/,about-us,about-us,4,45303,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
6,200,https://nesta.org.uk/toolkit/theory-change/,/toolkit/theory-change/,toolkit-theory-change,toolkit,14,16323,200,[toolkit-theory-change_Resources_2017_version_...,"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,True
7,200,https://nesta.org.uk/project/impact-investments/,/project/impact-investments/,project-impact-investments,project,10,23221,200,[],"[{'areasOfWork': 'Education,Health', 'missions...",NaN,False
8,200,https://nesta.org.uk/introduction-our-strategy/,/introduction-our-strategy/,introduction-our-strategy,introduction-our-strategy,8,23743,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False
9,200,https://nesta.org.uk/fairer-start/,/fairer-start/,fairer-start,fairer-start,9,23400,200,[],"[{'areasOfWork': '', 'missions': '', 'projects...",NaN,False


In [44]:
n = 19
uid = df.iloc[n].uid
print(uid)

# Load txt file as text string
text = open(PROJECT_DIR / f'data/outputs_2024-10-29/{uid}.txt').read()
print(scrape._scrape(text))

toolkit-collective-intelligence-design-playbook
This playbook was designed by Nesta to help you design and deliver a collective intelligence project.

Collective intelligence is created when people work together, often with the help of technology, to mobilise a wider range of information, ideas and insights to address a social challenge.

Although people have been working together since the dawn of time, collective intelligence has evolved quickly since the start of the digital age.

This playbook will introduce you to activities you can use to orchestrate diverse groups of people, data and technology to achieve your goals.

We call this collective intelligence design.

The playbook contains all of the content you need to design a collective intelligence project. It provides an introduction to collective intelligence and illustrative case studies. It will help you identify when and how to use collective intelligence design, and explain the stages in the collective intelligence design p

In [61]:
pdf_file = df.query("has_pdf").iloc[-1].pdf_files[0]
print(pdf_file)
df.query("has_pdf").iloc[-1].pdf_links[0]

report-the-future-of-skills-employment-in-2030_the_future_of_skills_employment_in_2030_0.pdf


'https://nesta.org.uk/documents/585/the_future_of_skills_employment_in_2030_0.pdf'

In [62]:
import pymupdf
pdf_path = PROJECT_DIR / f'data/outputs_2024-10-29/pdf_files/{pdf_file}'
doc = pymupdf.open(pdf_path)
pages = [page.get_text() for page in doc]

In [63]:
len(pages)

124

In [65]:
# pages[20]